# Effectiveness of Column Transformer!!!

___
### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df['cough'].value_counts() #Ordinal Data

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [ ]:
df['city'].value_counts() #Nominal Data

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
df.isnull().sum() #Fever column contains null values

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [9]:
X_train.head()

,age,gender,fever,cough,city
5,84,Female,NaN,Mild,Bangalore
37,55,Male,100.0,Mild,Kolkata
47,18,Female,104.0,Mild,Bangalore
27,33,Female,102.0,Strong,Delhi
53,83,Male,98.0,Mild,Delhi


___
## 1. Without using Column Transformer

___
### Importing SimpleImputer

SimpleImputer is used to handle missing values in a dataset.

It replaces missing values with a specified value or a statistical measure such as the mean, median, or most frequent value.

In [15]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [16]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [13]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [11]:
# Extracting Age
X_train_age = X_train[['age']].values

# also the test data
X_test_age = X_test[['age']].values

X_train_age.shape

(80, 1)

In [17]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

___
## With using Column Transformer

___
### Importing ColumnTransformer

ColumnTransformer is used to apply different preprocessing or transformation techniques to different columns of a dataset.

It is useful when a dataset contains different types of features that require different preprocessing methods.

For example, we can apply StandardScaler to numerical columns and OneHotEncoder to categorical columns at the same time.

In [18]:
from sklearn.compose import ColumnTransformer

___
### Creating a ColumnTransformer

In this code, three different transformers are created:

- tnf1 uses SimpleImputer on the fever column to handle missing values.
- tnf2 uses OrdinalEncoder on the cough column, where Mild is assigned a lower value than Strong.
- tnf3 uses OneHotEncoder on the gender and city columns to convert categorical values into numerical columns. drop='first' removes the first category to avoid redundant columns.
- remainder='passthrough' keeps all other columns unchanged.


tnf1, tnf2 and tnf3 are just names of the transformers.

This allows multiple preprocessing techniques to be applied to different columns within a single ColumnTransformer.

In [19]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [20]:
transformer.fit_transform(X_train).shape

(80, 7)

In [21]:
transformer.transform(X_test).shape

(20, 7)

In [23]:
X_train= transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)

In [27]:
X_train=pd.DataFrame(X_train)

In [28]:
X_train

,0,1,2,3,4,5,6
0,100.783784,0.0,0.0,0.0,0.0,0.0,84.0
1,100.000000,0.0,1.0,0.0,1.0,0.0,55.0
2,104.000000,0.0,0.0,0.0,0.0,0.0,18.0
3,102.000000,1.0,0.0,1.0,0.0,0.0,33.0
4,98.000000,0.0,1.0,1.0,0.0,0.0,83.0
...,...,...,...,...,...,...,...
75,103.000000,0.0,0.0,0.0,0.0,0.0,16.0
76,104.000000,0.0,0.0,0.0,0.0,0.0,12.0
77,101.000000,0.0,1.0,1.0,0.0,0.0,42.0
78,101.000000,0.0,0.0,0.0,1.0,0.0,8.0


___
### Thats all from Column Transformer Section !!!
___